# PyTorch model

In [1]:
import os
import torch
import sys
from torch import nn
# import lightning as L
import pandas as pd
import xarray as xr
import csv
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

In [2]:
import numpy as np
import matplotlib.pyplot as plt
# from brokenaxes import brokenaxes
import warnings 
import tqdm
warnings.filterwarnings("ignore")
from sunraster.instr.spice import read_spice_l2_fits # type: ignore


In [3]:
from simspice.data.SproutDataset_NeMg import SproutDataset
from torch.utils.data import DataLoader
from simspice.utils.Augmentation import Augmentation
from simspice.data.Sprout_ML_ONeS import Sprout_ML, interpolate_arrays
from simspice.data.Sprout_ML import Sprout_ML as sprml

In [4]:
import wandb
import astropy.units as u

In [5]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.device(0))
torch.cuda.get_device_name(0)
torch.cuda.set_device(2)

True
4


In [6]:
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [7]:
file = read_spice_l2_fits('/d0/tvaresano/SimSPICE/data_L2/SPICE_data_Jan2023/solo_L2_spice-n-ras_20230101T033421_V22_167772212-000.fits', memmap=False)

In [8]:
sprout = Sprout_ML(dir = '/d0/tvaresano/SimSPICE/data_L2/SPICE_data_Jan2023/', filename = 'solo_L2_spice-n-ras_20230101T033421_V22_167772212-000.fits')

In [ ]:
print(file['O III 703 / Mg IX 706 (Merged)'].spectral_axis.to(u.Angstrom).value)
print(file['S V 786 / O IV 787 (Merged)'].spectral_axis.to(u.Angstrom).value)
print(file['Ne VIII 770 (Merged)'].spectral_axis.to(u.Angstrom).value)

### Write to netcdf format 

In [9]:
import re
from datetime import datetime

def get_file_datetime(filename):
    """
    Extract a timestamp such as 20221229T033422 from a filename.
    """
    match = re.search(r"\d{8}T\d{6}", filename)
    if match is None:
        raise ValueError(f"No YYYYMMDDTHHMMSS timestamp found in: {filename}")
    return datetime.strptime(match.group(), "%Y%m%dT%H%M%S")

In [10]:
def write_files_to_netcdf(directory, netcdf_file, nbr_files=None):
    all_flux = []
    all_mask = []
    all_k = []
    all_l = []
    index = []
    filenames_ = []
    
    i = 0
    filenames = [filename for filename in os.listdir(directory) if os.path.isfile(os.path.join(directory, filename))]
    filenames_sorted = sorted(filenames, key=get_file_datetime)

    if nbr_files is not None:
        filenames_sorted = filenames_sorted[:nbr_files]

    for file_index, filename in enumerate(tqdm.tqdm(filenames_sorted)):
        print(file_index, get_file_datetime(filename), filename)

        if os.path.isfile(os.path.join(directory, filename)):
            print(filename)
            file = Sprout_ML(directory, filename)
            wvl_common = file.common_wvl
            
            #################################
            padded_spectra, masks = file.pad_flux_array(method='zeros')  # get padded flux arrays and masks
            wvl_arrays_padded = file.pad_wvl_arrays()
            full_spectra = np.vstack(padded_spectra)  # combine spectra into a single array
            full_mask = np.vstack(masks)  # combine masks
            full_wvl = np.hstack(wvl_arrays_padded)

            for k in range(full_spectra.shape[1]):
                for l in range(full_spectra.shape[2]):
                    spec_common_wvl, mask_common_wvl = interpolate_arrays(wvl_common, full_wvl, full_spectra[:, k, l], full_mask[:, k, l])
                    all_flux.append(spec_common_wvl)
                    all_mask.append(mask_common_wvl)
                    all_k.append(k)
                    all_l.append(l)
                    index.append(i)
                    filenames_.append(filename)
                    i += 1

    # xarray Dataset
    ds = xr.Dataset(
        {
            "flux": (("index", "wvl"), np.array(all_flux)),
            "mask": (("index", "wvl"), np.array(all_mask)),
            "x-index" : (("index"), all_k),
            "y-index" : (("index"), all_l),
            "filename" : (("index"), filenames_),
        },
        coords={"index": index,
                "wvl" : wvl_common, # interp1d array set of wvl every single spectrum is going to be interpolated into.
                }
            )

    # Save to NetCDF
    ds.to_netcdf(netcdf_file)

In [11]:
directory_path = '/d0/tvaresano/SimSPICE/data_L2/SPICE_data_Jan2023/'
netcdf_file_path = '/d0/tvaresano/SimSPICE/spectra_Jan2023_ONeS.nc'

write_files_to_netcdf(directory_path, netcdf_file_path)

  0%|          | 0/63 [00:00<?, ?it/s]

0 2022-12-25 03:34:21 solo_L2_spice-n-ras_20221225T033421_V22_150995989-000.fits
solo_L2_spice-n-ras_20221225T033421_V22_150995989-000.fits


  2%|▏         | 1/63 [00:11<11:55, 11.54s/it]

1 2022-12-25 15:34:21 solo_L2_spice-n-ras_20221225T153421_V22_150995993-000.fits
solo_L2_spice-n-ras_20221225T153421_V22_150995993-000.fits


  3%|▎         | 2/63 [00:23<11:51, 11.67s/it]

2 2022-12-26 03:34:21 solo_L2_spice-n-ras_20221226T033421_V22_167772163-000.fits
solo_L2_spice-n-ras_20221226T033421_V22_167772163-000.fits


... (117 outputs left out)

61 2023-01-30 03:34:22 solo_L2_spice-n-ras_20230130T033422_V22_167772448-000.fits
solo_L2_spice-n-ras_20230130T033422_V22_167772448-000.fits


 98%|█████████▊| 62/63 [12:05<00:11, 11.64s/it]

62 2023-01-30 15:34:22 solo_L2_spice-n-ras_20230130T153422_V22_167772452-000.fits
solo_L2_spice-n-ras_20230130T153422_V22_167772452-000.fits


100%|██████████| 63/63 [12:17<00:00, 11.70s/it]


In [13]:
os.chdir('/d0/tvaresano/SimSPICE/')
netcdf_file_path = 'spectra_9Jan2023_ONeS.nc'
directory_path = '/d0/tvaresano/SimSPICE/data_L2/9Jan2023/'
write_files_to_netcdf(directory_path, netcdf_file_path, 1)

  0%|          | 0/1 [00:00<?, ?it/s]

0 2023-01-09 03:34:21 solo_L2_spice-n-ras_20230109T033421_V22_167772289-000.fits
solo_L2_spice-n-ras_20230109T033421_V22_167772289-000.fits


100%|██████████| 1/1 [00:11<00:00, 11.42s/it]


In [14]:
netcdf_file_path = '/d0/tvaresano/SimSPICE/spectra_Feb2023_ONeS.nc'
directory_path = '/d0/tvaresano/SimSPICE/data_L2/Feb2023/'
write_files_to_netcdf(directory_path, netcdf_file_path, 1)

  0%|          | 0/1 [00:00<?, ?it/s]

0 2023-02-11 15:34:21 solo_L2_spice-n-ras_20230211T153421_V22_167772557-000.fits
solo_L2_spice-n-ras_20230211T153421_V22_167772557-000.fits


100%|██████████| 1/1 [00:11<00:00, 11.43s/it]


# Test augmentations

In [ ]:
import simspice.utils.inverse_mapping_functions as imf

In [ ]:
os.getcwd()
os.chdir('/d0/tvaresano/SimSPICE/')

In [ ]:
imf.map_item_map(30245,
                 dataset= "spectra_Feb2023_NeMg.nc",
                 plot=True,
                 data_dir = 'data_L2/Feb2023/')
plt.show()
imf.map_item_map(30942,
                 dataset= "spectra_Feb2023_NeMg.nc",
                 plot=True,
                 data_dir = 'data_L2/Feb2023/')

In [ ]:
all_spectra = xr.open_dataset("/d0/tvaresano/SimSPICE/spectra_Feb2023_NeMg.nc")
item = all_spectra.isel(index=22575)
item1 = all_spectra.isel(index=30245)

In [ ]:
aug = Augmentation(type_distrib_gain='gaussian', shift_range=(-0.04,0.04))
augmented1 = aug.run_all_augmentations(item)[0]
augmented2 = aug.run_all_augmentations(item)[0]

In [ ]:
plt.figure(figsize=[7,4], tight_layout=True)
plt.plot(item['wvl'].values, item['flux'].values, label='original', alpha=1, color='k')
plt.plot(item['wvl'].values, augmented1, label='augmentation 1')
plt.plot(item['wvl'].values, augmented2, label='augmentation 2')
plt.xlabel('Wavelength ($\AA$)')
# plt.plot(item['wvl'].values, item1['flux'].values, label='spectrum1', alpha=0.5)
# plt.axhline(np.median(item1['flux'].values))
plt.legend()
plt.title('Ne VIII and Mg VIII')

In [ ]:
item = all_spectra.isel(index=22575)
item1 = all_spectra.isel(index=30250)
item2 = all_spectra.isel(index=30242)

In [ ]:
plt.figure(figsize=[10,4], tight_layout=True)
plt.plot(item['wvl'].values, item['flux'].values, color='k')
# plt.plot(item['wvl'].values, item1['flux'].values, label='spectrum')
# plt.plot(item['wvl'].values, item2['flux'].values, label='spectrum')
plt.xlabel('Wavelength ($\AA$)')
# plt.plot(item['wvl'].values, item1['flux'].values, label='spectrum1', alpha=0.5)
# plt.axhline(np.median(item1['flux'].values))
plt.legend()
plt.title('Ne VIII and Mg VIII raster')